<a href="https://colab.research.google.com/github/matrixportalx/Sd-1.5-Converting-to-Qualcomm-QNN-Model/blob/notebook/SD15_NPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SD 1.5 → Qualcomm NPU (Local Dream / Ruya) — RESMİ HAT

Bu defter, Local Dream'in **kendi dönüştürme scriptlerini** (`npuconvertv2`)
QNN SDK **2.28** ile koşar. Rehber: `ld-guide.chino.icu/conversion/sd15`

**Neden bu hat:** kendi yazdığımız hat cihazda yüklenmeyen paketler üretiyordu.
Sebepleri resmi scriptlerde görüldü:

| bizim (eski) | resmi |
|---|---|
| `qairt-converter` → DLC | `qnn-onnx-converter` → `model.cpp` → `.so` |
| `--act_bitwidth 8` + io16 hilesi | **`--act_bitwidth 16`** |
| per-channel kapalı | `--use_per_channel_quantization` |
| stok diffusers | `redefined_modules/` (MHA→SHA, Linear→Conv) |
| VTCM ayarsız | `"vtcm_mb": 2` |

**Çalışma sırası:** 1 → 2 → 3 → 4 → 5

> **GPU notu:** GPU yalnızca kalibrasyon verisi üretimini (`prepare_data.py`)
> hızlandırır — ~35 dk yerine ~3 dk. Kuantizasyon, model-lib ve context-binary
> adımları **tamamen CPU**'dur ve GPU'dan etkilenmez. Yine de yüksek RAM
> gerektiği için (rehber: ~20 GB) yüksek bellekli çalışma zamanı şart.


## 1) Ayarlar

In [ ]:
#@title Ayarlar { display-mode: "form" }
#@markdown Model dosyasının **doğrudan indirme** bağlantısı (.safetensors)
SAFETENSORS_URL = "https://civitai.com/api/download/models/2681234?fileId=2567874"  #@param {type:"string"}
MODEL_NAME = "CyberRealistic_qnn2.28_min"  #@param {type:"string"}
#@markdown Çip katmanı — `min` = Hexagon V68+ (Snapdragon 7 Gen 1 dahil)
SOC = "min"  #@param ["min", "8gen1", "8gen2"]
#@markdown `clip_skip`: modelin eğitildiği değer. Anime çoğunlukla 2.
CLIP_SKIP = 2  #@param [1, 2] {type:"raw"}
#@markdown Foto-gerçekçi model ise işaretleyin (kalibrasyon promptlarını değiştirir)
REALISTIC = True  #@param {type:"boolean"}
#@markdown Kuantizasyon örnek sayısı. Resmi tarif 400 (saatler).
#@markdown `24` = boru hattını doğrula, `150` = iyi denge, `0` = tam (400)
CALIB_LIMIT = 150  #@param {type:"integer"}
#@markdown GPU varsa CUDA torch kur (prepare_data ~10x hızlanır)
CUDA_TORCH = True  #@param {type:"boolean"}

import os
os.environ.update(
    MODEL_NAME=MODEL_NAME, SOC=SOC,
    CLIP_SKIP=str(CLIP_SKIP),
    REALISTIC="1" if REALISTIC else "0",
    CALIB_LIMIT=str(CALIB_LIMIT),
    CUDA_TORCH="1" if CUDA_TORCH else "0",
    SAFETENSORS_URL=SAFETENSORS_URL,
)
print(f"{MODEL_NAME} | soc={SOC} clip_skip={CLIP_SKIP} realistic={REALISTIC}")
print(f"calib_limit={CALIB_LIMIT} cuda_torch={CUDA_TORCH}")
!nvidia-smi -L || echo "GPU yok — prepare_data yavas olacak"


## 2) Depo + araçlar

Depoyu klonlar/günceller ve `uv`'yi kurar. Çalışma zamanı sıfırlansa bile
tekrar çalıştırmak yeterli.

In [ ]:
%cd /content
REPO = "https://github.com/matrixportalx/Sd-1.5-Converting-to-Qualcomm-QNN-Model"
BRANCH = "dev"
import os, subprocess
if not os.path.isdir("/content/sd-qnn/.git"):
    !git clone -b {BRANCH} {REPO} /content/sd-qnn
else:
    !cd /content/sd-qnn && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}
%cd /content/sd-qnn
!pip install -q uv
!git log --oneline -1


1296962 (HEAD -> dev, dev, origin/HEAD) Indirme mantigini scripts/fetch_ckpt.py'ye tasi (kisa hucre)


## 3) QNN SDK 2.28

~2 GB. **Sürüm önemli** — rehber 2.28 şart koşuyor. İndirme koparsa bu hücreyi
tekrar çalıştırın, kaldığı yerden devam eder.

In [ ]:
%cd /content/sd-qnn
import os
URL = ("https://apigwx-aws.qualcomm.com/qsc/public/v1/api/download/software/"
       "qualcomm_neural_processing_sdk/v2.28.0.241029.zip")
out = !python3 scripts/setup_qnn_sdk.py --dest /content/qairt --asset-url "{URL}"
print("\n".join(out[-25:]))
root = [l.split("=",1)[1] for l in out if l.startswith("QNN_SDK_ROOT=")]
assert root, "QNN_SDK_ROOT bulunamadi — yukaridaki ciktiya bakin"
os.environ["QNN_SDK_ROOT"] = root[-1].strip()
print("\nQNN_SDK_ROOT =", os.environ["QNN_SDK_ROOT"])


/content/sd-qnn
    935/954 MB (97%)
    936/954 MB (98%)
    937/954 MB (98%)
    938/954 MB (98%)
    939/954 MB (98%)
    940/954 MB (98%)
    941/954 MB (98%)
    942/954 MB (98%)
    943/954 MB (98%)
    944/954 MB (98%)
    945/954 MB (98%)
    946/954 MB (99%)
    947/954 MB (99%)
    948/954 MB (99%)
    949/954 MB (99%)
    950/954 MB (99%)
    951/954 MB (99%)
    952/954 MB (99%)
    953/954 MB (99%)
    954/954 MB (99%)
    954/954 MB (100%)
[*] Aciliyor -> /content/qairt/2.28.0.241029
[*] 180 dosyaya calistirma izni verildi (bin/)
[+] QNN_SDK_ROOT = /content/qairt/2.28.0.241029/qairt/2.28.0.241029
QNN_SDK_ROOT=/content/qairt/2.28.0.241029/qairt/2.28.0.241029

QNN_SDK_ROOT = /content/qairt/2.28.0.241029/qairt/2.28.0.241029


## 4) Modeli indir

Resmi hat `.safetensors` dosyasını **doğrudan** kullanır.

In [ ]:
%cd /content/sd-qnn
import os
try:
    from google.colab import userdata
    for k in ("CIVITAI_TOKEN", "HF_TOKEN"):
        v = userdata.get(k)
        if v: os.environ[k] = v
except Exception as e:
    print("[!] Secrets:", e)
!python scripts/fetch_ckpt.py --out work/input.safetensors

/content/sd-qnn
[*] indiriliyor: https://civitai.com/api/download/models/2681234
    2033 / 2033 MB (100%)
[+] 2.13 GB -> work/input.safetensors


## 5) Dönüştür

Aşamalar: `uv` ortamı → `prepare_data` → `gen_quant_data` → `export_onnx` →
`qnn-onnx-converter` → `qnn-model-lib-generator` → `qnn-context-binary-generator`

`data.pkl` ve `unet/model.onnx` önbelleğe alınır: `CALIB_LIMIT` değiştirip
tekrar çalıştırırsanız veri üretimi **atlanır**, sadece kuantizasyon yenilenir.

In [ ]:
%cd /content/sd-qnn
import os
assert os.environ.get("QNN_SDK_ROOT"), "Once 3. hucreyi calistirin"
name = os.environ["MODEL_NAME"]
!bash scripts/06_official_pipeline.sh work/input.safetensors "{name}" "work/{name}" "$SOC"


/content/sd-qnn
### resmi scriptler indiriliyor (npuconvertv2)
[*] rehber taraniyor: https://ld-guide.chino.icu/conversion/sd15
[*] bulunan .zip linkleri: ['https://apigwx-aws.qualcomm.com/qsc/public/v1/api/download/software/qualcomm_neural_processing_sdk/v2.28.0.241029.zip', 'https://chino.icu/local-dream/npuconvertv2.zip']
[*] indiriliyor: https://chino.icu/local-dream/npuconvertv2.zip
[+] 7.3 MB -> work/CyberRealistic_qnn2.28_min/_official/npuconvert.zip
[+] 62 dosya acildi
[+] 8 dosyaya calistirma izni verildi

--- ICERIK ---
    npuconvertv2/
    npuconvertv2/.gitignore
    npuconvertv2/MNNConvert
    npuconvertv2/export.sh
    npuconvertv2/export_onnx.py
    npuconvertv2/export_onnx_unet_only.py
    npuconvertv2/gen_quant_data.py
    npuconvertv2/htp_backend_8gen1.json
    npuconvertv2/htp_backend_8gen2.json
    npuconvertv2/htp_backend_min.json
    npuconvertv2/htp_config_8gen1.json
    npuconvertv2/htp_config_8gen2.json
    npuconvertv2/htp_config_min.json
    npuconvertv2/prep

## 6) Paketi indir

`BITTI` satırını gördükten sonra çalıştırın. **Hemen indirin** — çalışma zamanı
kapanırsa dosya kaybolur.

In [ ]:
import glob, os
from google.colab import files
zips = sorted(glob.glob("/content/sd-qnn/dist/*.zip"), key=os.path.getmtime)
assert zips, "dist/ bos — donusum tamamlanmadi"
z = zips[-1]
print(f"{z}  ({os.path.getsize(z)/1e6:.0f} MB)")
!unzip -l "{z}"
files.download(z)


## 7) Hugging Face'e yükle (isteğe bağlı)

Telefona indirmek yerine (ya da ek olarak) ZIP'i HF'e koyar: çalışma zamanı
kapansa bile kalıcı olur ve telefondan doğrudan indirilebilir.

**Gerekli:** yazma (write) izinli token → Colab sol menü **🔑 Secrets →
`HF_TOKEN`** (*Notebook access* açık olmalı).

Alanlar bu hücrenin kendi formunda — başka hücreye bağımlı değil.


In [ ]:
#@title Hugging Face'e yükle { display-mode: "form" }
#@markdown **Ayri repo:** her model `<kullanici>/<MODEL_NAME>` reposuna gider.
#@markdown **Koleksiyon:** hepsi tek repoda, her model kendi alt klasorunde.
UPLOAD_MODE = "Ayri repo"  #@param ["Ayri repo", "Koleksiyon"]
#@markdown Koleksiyon modunda kullanilacak repo adi
COLLECTION_REPO = "sd_qnn"  #@param {type:"string"}
#@markdown Elle tam repo adi (`kullanici/repo`) — doluysa yukaridakiler yok sayilir
HF_REPO = ""  #@param {type:"string"}
HF_PRIVATE = False  #@param {type:"boolean"}

import glob, os, subprocess, sys
os.chdir("/content/sd-qnn")
assert os.path.isdir("scripts"), "Once 2. adimi (depoyu cek) calistirin!"

zips = sorted(glob.glob("dist/*.zip"), key=os.path.getmtime)
assert zips, "dist/ icinde zip yok — 5. adim (donusum) tamamlanmamis olabilir."
zip_path = zips[-1]
print(f"{zip_path}  ({os.path.getsize(zip_path)/1e6:.0f} MB)")

# Token: Colab Secrets -> HF_TOKEN (write izinli olmali)
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
except Exception as e:
    print("[!] Secrets okunamadi:", e)
assert os.environ.get("HF_TOKEN"), \
    "HF_TOKEN yok — Colab Secrets (anahtar simgesi) -> HF_TOKEN ekleyin (write)."

try:
    import huggingface_hub  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "huggingface_hub"], check=True)

# 1. hucre calistirilmadiysa model adini zip isminden turet
name = os.environ.get("MODEL_NAME", "").strip() or \
    os.path.basename(zip_path).split("_qnn")[0]

cmd = [sys.executable, "scripts/upload_hf.py", "--file", zip_path,
       "--name", name]
if HF_REPO.strip():
    cmd += ["--repo", HF_REPO.strip()]
elif UPLOAD_MODE == "Koleksiyon":
    cmd += ["--collection", COLLECTION_REPO.strip()]
if HF_PRIVATE:
    cmd.append("--private")

print(">", " ".join(cmd))
rc = subprocess.run(cmd).returncode
assert rc == 0, f"Yukleme basarisiz (cikis kodu {rc}) — yukaridaki hataya bakin."
